In [1]:
import os
import shutil


In [2]:
data_folder = 'data'
os.makedirs(data_folder, exist_ok=True)

# Iteriere über alle Ordner im aktuellen Verzeichnis
for root, dirs, files in os.walk('.'):
    for dir_name in dirs:
        # Überprüfen, ob "batch" im Ordnernamen enthalten ist
        if 'batch' in dir_name:
            batch_folder_path = os.path.join(root, dir_name)
            # Durchsuchen der Dateien in diesem Ordner
            for file_name in os.listdir(batch_folder_path):
                # Überprüfen, ob die Datei eine .txt oder .ann Datei ist
                if file_name.endswith('.txt') or file_name.endswith('.ann'):
                    file_path = os.path.join(batch_folder_path, file_name)
                    shutil.move(file_path, data_folder)
                    print(f"Moved {file_path} to {data_folder}")

## Erzeuge IC und EC getrennt

In [10]:
import os
import re

def read_criteria_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

def write_to_file(file_path, content):
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(content)

def process_files(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.endswith(".txt"):
            nct_number = re.findall(r'NCT\d+', filename)
            if not nct_number:
                continue
            nct_number = nct_number[0]
            file_path = os.path.join(input_folder, filename)
            content = read_criteria_file(file_path)

            lines = content.split('\n')
            inclusion_criteria = []
            exclusion_criteria = []
            inclusion_subsection = None
            exclusion_subsection = None
            current_section = None

            for line in lines:
                line_lower = line.lower().strip()
                if "inclusion criteria" in line_lower:
                    current_section = "inclusion"
                    inclusion_subsection = line.strip()
                elif "exclusion criteria" in line_lower:
                    current_section = "exclusion"
                    exclusion_subsection = line.strip()
                elif current_section == "inclusion":
                    if line.strip().startswith("-"):
                        inclusion_criteria.append(line.strip())
                    else:
                        inclusion_criteria.append(inclusion_subsection)
                        inclusion_criteria.append(line.strip())
                        inclusion_subsection = None
                elif current_section == "exclusion":
                    if line.strip().startswith("-"):
                        exclusion_criteria.append(line.strip())
                    else:
                        exclusion_criteria.append(exclusion_subsection)
                        exclusion_criteria.append(line.strip())
                        exclusion_subsection = None

            # Remove empty lines and duplicates
            inclusion_criteria = [line for line in inclusion_criteria if line and not line.startswith("Inclusion Criteria")]
            exclusion_criteria = [line for line in exclusion_criteria if line and not line.startswith("Exclusion Criteria")]

            if inclusion_criteria:
                inc_filename = f"{nct_number}_inc.txt"
                inc_file_path = os.path.join(output_folder, inc_filename)
                write_to_file(inc_file_path, "\n".join(inclusion_criteria))

            if exclusion_criteria:
                exc_filename = f"{nct_number}_exc.txt"
                exc_file_path = os.path.join(output_folder, exc_filename)
                write_to_file(exc_file_path, "\n".join(exclusion_criteria))

# Beispielaufruf
input_folder = "data"
output_folder = "data_half"
process_files(input_folder, output_folder)


In [6]:
# NCT03860181